# 위성 시계열 데이터 기반 Semi-Supervised Learning 실험

세 논문(CGMatch, InterLUDE, SC-SSL)이 각자 완화한 가정의 위배 조건에서 고전 기법(Self-training)보다 실제로 더 기여하는지 본다.

- 방법 6종 × 조건 3종 × seed 3개 = 54회 학습
- 로직은 `sslexp/*.py`에 있고 이 노트북은 호출만 한다. 설명은 `README.md`.
- 끊기면 같은 셀을 다시 실행한다(끝난 실행은 건너뛴다).

In [ ]:
# 0. 준비: Drive 연결, 경로, 코드 불러오기
from google.colab import drive
drive.mount('/content/drive')
import os, sys, importlib, torch

DATA     = "/content/drive/MyDrive/AI4Biodynamics/data"            # 읽기만 한다
CODE_DIR = "/content/drive/MyDrive/ssl_seminar/ssl_experiment"     # sslexp 폴더가 바로 아래에 있는 위치
OUT      = "/content/drive/MyDrive/ssl_seminar/results_v1"         # 설정을 바꾸면 새 폴더로

if not os.path.isdir(f"{CODE_DIR}/sslexp"):
    found = [r for r, d, _ in os.walk("/content/drive/MyDrive") if "sslexp" in d][:5]
    raise FileNotFoundError(f"{CODE_DIR} 아래에 sslexp 폴더가 없습니다. CODE_DIR을 고치세요. Drive에서 찾은 위치: {found if found else '없음(폴더를 아직 올리지 않았습니다)'}")
sys.path.insert(0, CODE_DIR)
import sslexp.data as D, sslexp.methods as M, sslexp.run as R
D, M, R = importlib.reload(D), importlib.reload(M), importlib.reload(R)
HW = R.hw_profile()      # GPU, vCPU, RAM 확인
if not HW["cuda"]: print("GPU가 없습니다. 런타임 유형을 GPU로 바꾸세요.")

In [ ]:
# 1. 데이터 준비와 규모 확인 (학습을 시작하기 전에 이 출력을 본다)
MIN_TRAIN = 300      # 학습 구역 지점이 이보다 적은 클래스는 뺀다(100보다 커야 한다)
bundle = R.prepare(csv=f"{DATA}/HLS_TimeSeries_2022.csv", dense_cache=f"{DATA}/hls_dense_cache.pt", xy_cache=f"{DATA}/site_xy.npy",
                   work_dir=OUT, split="block", block_km=25.0, min_train=MIN_TRAIN)
print()
D.describe_conditions(bundle, ("base", "A", "B"))

In [ ]:
# (선택) 학습을 시작하지 않고 예상 소요 시간만 본다. 끝난 실행이 있으면 그 기록으로, 없으면 20스텝 시험 학습으로 계산한다.
_ = R.estimate_time(bundle, OUT, cfg=M.Cfg(iters=6000))

In [ ]:
# 2. 주 실험. 시작할 때 런타임과 예상 소요 시간이 출력되고, 도는 동안 진행 막대가 보인다.
#    맨 위 막대 = 전체 진행(경과 시간 · 남은 시간), 그 아래 = 워커별로 지금 도는 실행(스텝 · val F1)
#    실행이 하나 끝날 때마다 결과 한 줄과 다시 계산한 남은 예상 시간이 찍힌다.
#    처음에는 아래 두 줄을 SEEDS = [0], CONDITIONS = ["A"]로 바꿔 한 묶음만 돌려 본다.
SEEDS      = [0, 1, 2]
CONDITIONS = ["base", "A", "B"]
METHODS    = list(M.METHODS)          # supervised, pseudolabel, fixmatch, cgmatch, interlude, scssl
WORKERS    = "auto"                   # 동시에 돌릴 학습 수. "auto" = vCPU 수와 GPU 여유 메모리에 맞춤, 1 = 차례로
cfg = M.Cfg(iters=6000)

R.run_grid(bundle, OUT, methods=METHODS, conditions=CONDITIONS, seeds=SEEDS, cfg=cfg, workers=WORKERS)

In [ ]:
# 3. 결과 표 (OUT/tables 에도 저장된다)
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
T = R.tables(OUT)
for name, t in T.items():
    print(f"\n== {name}"); display(t)

## 선택: 조건 Bc (B의 대조군)

조건 B는 기준 조건과 두 가지가 다르다(labeled가 long-tail, unlabeled 분포가 반대). `Bc`는 labeled는 B와 같고 unlabeled만 같은 방향으로 둔 조건이라, B와 Bc를 비교하면 "분포 불일치"만의 영향을 볼 수 있다. 18회 학습이 추가된다.

In [ ]:
# R.run_grid(bundle, OUT, methods=METHODS, conditions=["Bc"], seeds=SEEDS, cfg=cfg)
# for name, t in R.tables(OUT).items(): print(f"\n== {name}"); display(t)